# Exploratory Data Analysis — TCGA-LUAD Lung Adenocarcinoma

Multi-omics dataset containing three normalised modalities:
- **RNASeq** — gene expression (447 samples × 14 434 genes)
- **DNAm** — DNA methylation β-values (420 samples × 384 629 CpG sites)
- **CNV** — copy-number variation (424 samples × 14 434 genes)
- **metadata** — sample annotations (846 entries)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = 'Group2-TCGA-LUAD-lung-adeno/'
print('Libraries loaded.')

Libraries loaded.


## 1. Load data

In [ ]:
meta   = pd.read_csv(DATA_DIR + 'metadata.csv')
rnaseq = pd.read_csv(DATA_DIR + 'RNASeq.csv', index_col=0)
dnam   = pd.read_csv(DATA_DIR + 'DNAm.csv',   index_col=0)
cnv    = pd.read_csv(DATA_DIR + 'CNV.csv',    index_col=0)

print('metadata :', meta.shape)
print('RNASeq   :', rnaseq.shape)
print('DNAm     :', dnam.shape)
print('CNV      :', cnv.shape)

## 2. Metadata overview

In [ ]:
meta.head()

In [ ]:
print(meta.dtypes)
print('\nMissing values:')
print(meta.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Label distribution
label_counts = meta['label'].value_counts()
axes[0].bar(label_counts.index.astype(str), label_counts.values, color=['steelblue','salmon'])
axes[0].set_title('Label distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

# Sample type
st_counts = meta['sample_type'].value_counts()
axes[1].barh(st_counts.index, st_counts.values, color='teal')
axes[1].set_title('Sample types')
axes[1].set_xlabel('Count')

# Data availability per modality
avail = {
    'RNASeq': meta['has_RNASeq'].sum(),
    'DNAm':   meta['has_DNAm'].sum(),
    'CNV':    meta['has_CNV'].sum()
}
axes[2].bar(avail.keys(), avail.values(), color=['#4c72b0','#dd8452','#55a868'])
axes[2].set_title('Samples per modality')
axes[2].set_ylabel('Count')
for i, v in enumerate(avail.values()):
    axes[2].text(i, v + 1, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Samples with all three modalities
all3 = meta[meta['has_RNASeq'] & meta['has_DNAm'] & meta['has_CNV']]
print(f'Samples with all 3 modalities: {len(all3)}')
print('\nLabel breakdown (all-3):')
print(all3['label'].value_counts())

## 3. RNASeq — gene expression

In [ ]:
print('Shape:', rnaseq.shape)
print('\nValue range:')
print(rnaseq.stack().describe())

In [ ]:
print('Missing values:', rnaseq.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of all values
flat_rna = rnaseq.values.ravel()
axes[0].hist(flat_rna, bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('RNASeq value distribution')
axes[0].set_xlabel('Normalised expression')
axes[0].set_ylabel('Count')

# Per-sample mean expression
sample_means = rnaseq.mean(axis=1)
axes[1].hist(sample_means, bins=40, color='steelblue', edgecolor='none')
axes[1].set_title('Per-sample mean expression')
axes[1].set_xlabel('Mean')

# Gene variance distribution (top 1000)
gene_var = rnaseq.var(axis=0).sort_values(ascending=False)
axes[2].plot(range(len(gene_var[:5000])), gene_var[:5000].values, color='steelblue')
axes[2].set_title('Gene variance (top 5000)')
axes[2].set_xlabel('Gene rank')
axes[2].set_ylabel('Variance')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most variable genes
print('Top 10 most variable genes:')
print(gene_var.head(10).to_string())

In [ ]:
# PCA on top 2000 most variable genes
top_genes = gene_var.head(2000).index
rna_sub   = rnaseq[top_genes]

scaler = StandardScaler()
rna_scaled = scaler.fit_transform(rna_sub)

pca_rna = PCA(n_components=20, random_state=42)
rna_pcs = pca_rna.fit_transform(rna_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, 21), pca_rna.explained_variance_ratio_ * 100, color='steelblue')
axes[0].plot(range(1, 21), np.cumsum(pca_rna.explained_variance_ratio_) * 100,
             color='red', marker='o', markersize=4, label='Cumulative')
axes[0].set_title('RNASeq PCA — explained variance')
axes[0].set_xlabel('PC')
axes[0].set_ylabel('% variance explained')
axes[0].legend()

# PC1 vs PC2, coloured by sample index (no label merge yet)
sc = axes[1].scatter(rna_pcs[:, 0], rna_pcs[:, 1],
                     alpha=0.6, s=20, c=range(len(rna_pcs)), cmap='viridis')
axes[1].set_title(f'RNASeq PC1 vs PC2')
axes[1].set_xlabel(f'PC1 ({pca_rna.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_rna.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc, ax=axes[1], label='Sample index')

plt.tight_layout()
plt.show()

## 4. DNA Methylation

In [ ]:
print('Shape:', dnam.shape)
print('\nValue range (β-values expected in [0,1]):')
print(dnam.stack().describe())

In [ ]:
missing_cpg = dnam.isnull().sum()
pct_missing = (missing_cpg > 0).sum() / dnam.shape[1] * 100
print(f'CpG sites with any missing value: {(missing_cpg>0).sum()} ({pct_missing:.1f}%)')
print(f'Total missing entries: {dnam.isnull().sum().sum()}')

In [ ]:
# Sample a subset for speed (384 k columns → plot subset)
np.random.seed(42)
sample_cpg = np.random.choice(dnam.columns, size=50000, replace=False)
dnam_sub = dnam[sample_cpg]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

flat_dnam = dnam_sub.values.ravel()
flat_dnam = flat_dnam[~np.isnan(flat_dnam)]
axes[0].hist(flat_dnam, bins=80, color='#dd8452', edgecolor='none')
axes[0].set_title('DNAm β-value distribution (50k CpGs sample)')
axes[0].set_xlabel('β-value')
axes[0].set_ylabel('Count')

sample_mean_m = dnam.mean(axis=1)
axes[1].hist(sample_mean_m, bins=40, color='#dd8452', edgecolor='none')
axes[1].set_title('Per-sample mean β-value')
axes[1].set_xlabel('Mean β')

cpg_var = dnam_sub.var(axis=0).sort_values(ascending=False)
axes[2].plot(range(len(cpg_var)), cpg_var.values, color='#dd8452')
axes[2].set_title('CpG variance (50k sample, ranked)')
axes[2].set_xlabel('CpG rank')
axes[2].set_ylabel('Variance')

plt.tight_layout()
plt.show()

In [ ]:
# PCA on top 5000 most variable CpGs (from 50k sample)
top_cpgs   = cpg_var.head(5000).index
dnam_pca_in = dnam[top_cpgs].dropna(axis=1)  # drop any remaining NaN columns

dnam_scaled = StandardScaler().fit_transform(dnam_pca_in)
pca_dnam = PCA(n_components=20, random_state=42)
dnam_pcs = pca_dnam.fit_transform(dnam_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, 21), pca_dnam.explained_variance_ratio_ * 100, color='#dd8452')
axes[0].plot(range(1, 21), np.cumsum(pca_dnam.explained_variance_ratio_) * 100,
             color='red', marker='o', markersize=4, label='Cumulative')
axes[0].set_title('DNAm PCA — explained variance')
axes[0].set_xlabel('PC')
axes[0].set_ylabel('% variance explained')
axes[0].legend()

sc = axes[1].scatter(dnam_pcs[:, 0], dnam_pcs[:, 1],
                     alpha=0.6, s=20, c=range(len(dnam_pcs)), cmap='plasma')
axes[1].set_title('DNAm PC1 vs PC2')
axes[1].set_xlabel(f'PC1 ({pca_dnam.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_dnam.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc, ax=axes[1], label='Sample index')

plt.tight_layout()
plt.show()

## 5. Copy-Number Variation

In [ ]:
print('Shape:', cnv.shape)
print('\nValue summary:')
print(cnv.stack().describe())

In [ ]:
print('Missing values:', cnv.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

flat_cnv = cnv.values.ravel()
axes[0].hist(flat_cnv, bins=80, color='#55a868', edgecolor='none')
axes[0].set_title('CNV value distribution')
axes[0].set_xlabel('CNV score')
axes[0].set_ylabel('Count')

sample_mean_cnv = cnv.mean(axis=1)
axes[1].hist(sample_mean_cnv, bins=40, color='#55a868', edgecolor='none')
axes[1].set_title('Per-sample mean CNV')
axes[1].set_xlabel('Mean')

gene_var_cnv = cnv.var(axis=0).sort_values(ascending=False)
axes[2].plot(range(len(gene_var_cnv[:5000])), gene_var_cnv[:5000].values, color='#55a868')
axes[2].set_title('Gene CNV variance (top 5000)')
axes[2].set_xlabel('Gene rank')
axes[2].set_ylabel('Variance')

plt.tight_layout()
plt.show()

In [ ]:
# PCA on top 2000 most variable CNV genes
top_cnv_genes = gene_var_cnv.head(2000).index
cnv_sub       = cnv[top_cnv_genes]

cnv_scaled = StandardScaler().fit_transform(cnv_sub)
pca_cnv = PCA(n_components=20, random_state=42)
cnv_pcs = pca_cnv.fit_transform(cnv_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, 21), pca_cnv.explained_variance_ratio_ * 100, color='#55a868')
axes[0].plot(range(1, 21), np.cumsum(pca_cnv.explained_variance_ratio_) * 100,
             color='red', marker='o', markersize=4, label='Cumulative')
axes[0].set_title('CNV PCA — explained variance')
axes[0].set_xlabel('PC')
axes[0].set_ylabel('% variance explained')
axes[0].legend()

sc = axes[1].scatter(cnv_pcs[:, 0], cnv_pcs[:, 1],
                     alpha=0.6, s=20, c=range(len(cnv_pcs)), cmap='Greens')
axes[1].set_title('CNV PC1 vs PC2')
axes[1].set_xlabel(f'PC1 ({pca_cnv.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_cnv.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc, ax=axes[1], label='Sample index')

plt.tight_layout()
plt.show()

## 6. Cross-modality: sample overlap and PCA coloured by label

Harmonise sample IDs across modalities (RNASeq uses `.` separators, metadata uses `-`).

In [ ]:
# Normalise IDs: replace dots with dashes, keep only first 12 chars (TCGA patient ID)
def short_id(idx):
    return idx.str.replace('.', '-', regex=False).str[:12]

rna_ids  = short_id(rnaseq.index.to_series())
dnam_ids = short_id(dnam.index.to_series())
cnv_ids  = short_id(cnv.index.to_series())

# Map patient_id → label from metadata
label_map = meta.set_index('patient_id')['label']

rna_labels  = rna_ids.map(label_map).values
dnam_labels = dnam_ids.map(label_map).values
cnv_labels  = cnv_ids.map(label_map).values

print('RNASeq label coverage:',  (~pd.isna(rna_labels)).sum(), '/', len(rna_labels))
print('DNAm label coverage:',    (~pd.isna(dnam_labels)).sum(), '/', len(dnam_labels))
print('CNV label coverage:',     (~pd.isna(cnv_labels)).sum(), '/', len(cnv_labels))

In [ ]:
palette = {0: '#4c72b0', 1: '#dd8452'}

def pca_scatter(ax, pcs, labels, evr, title, cmap_dict):
    labels = np.array(labels, dtype=float)
    for val, col in cmap_dict.items():
        mask = labels == val
        ax.scatter(pcs[mask, 0], pcs[mask, 1],
                   alpha=0.65, s=22, c=col, label=f'Label {int(val)}')
    # plot unknowns
    unk = np.isnan(labels)
    if unk.any():
        ax.scatter(pcs[unk, 0], pcs[unk, 1], alpha=0.3, s=18, c='grey', label='Unknown')
    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pca_scatter(axes[0], rna_pcs,  rna_labels,  pca_rna.explained_variance_ratio_,  'RNASeq PCA',  palette)
pca_scatter(axes[1], dnam_pcs, dnam_labels, pca_dnam.explained_variance_ratio_, 'DNAm PCA',    palette)
pca_scatter(axes[2], cnv_pcs,  cnv_labels,  pca_cnv.explained_variance_ratio_,  'CNV PCA',     palette)
plt.suptitle('PC1 vs PC2 coloured by label (0 = Normal, 1 = Tumour)', y=1.02)
plt.tight_layout()
plt.show()

## 7. Top variable features — cross-modality correlation heatmap

Select the top 30 most variable genes from RNASeq and CNV (same gene space) and check their correlation.

In [ ]:
# Common genes between RNASeq and CNV
common_genes = list(set(rnaseq.columns) & set(cnv.columns))
print(f'Genes in common between RNASeq and CNV: {len(common_genes)}')

# Common samples (by patient ID)
rna_df  = rnaseq.copy(); rna_df.index  = rna_ids.values
cnv_df  = cnv.copy();    cnv_df.index  = cnv_ids.values

shared_samples = list(set(rna_df.index) & set(cnv_df.index))
print(f'Shared samples (patient IDs): {len(shared_samples)}')

rna_shared = rna_df.loc[shared_samples, common_genes]
cnv_shared = cnv_df.loc[shared_samples, common_genes]

In [ ]:
# Top 30 most variable genes in RNASeq (from the shared set)
top30 = rna_shared.var(axis=0).sort_values(ascending=False).head(30).index

# Correlation between RNASeq and CNV across samples for each gene
rna_cnv_corr = pd.DataFrame({
    gene: rna_shared[gene].corr(cnv_shared[gene]) for gene in top30
}, index=['RNA–CNV corr'])

fig, ax = plt.subplots(figsize=(14, 2))
sns.heatmap(rna_cnv_corr, ax=ax, cmap='coolwarm', center=0,
            linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('RNASeq–CNV per-gene correlation (top 30 variable genes)')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise sample correlation within RNASeq (top 30 genes) as a heatmap
sample_corr_rna = rna_shared[top30].T.corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(sample_corr_rna, ax=ax, cmap='coolwarm', center=0,
            xticklabels=False, yticklabels=False,
            cbar_kws={'label': 'Pearson r'})
ax.set_title(f'Sample–sample correlation (RNASeq, top-30 variable genes, {len(shared_samples)} shared samples)')
plt.tight_layout()
plt.show()

## 8. Summary table

In [ ]:
summary = pd.DataFrame({
    'Modality': ['RNASeq', 'DNAm', 'CNV'],
    'Samples': [rnaseq.shape[0], dnam.shape[0], cnv.shape[0]],
    'Features': [rnaseq.shape[1], dnam.shape[1], cnv.shape[1]],
    'Missing (total)': [
        rnaseq.isnull().sum().sum(),
        dnam.isnull().sum().sum(),
        cnv.isnull().sum().sum()
    ],
    'Value min': [rnaseq.values.min(), dnam.values.min(), cnv.values.min()],
    'Value max': [rnaseq.values.max(), dnam.values.max(), cnv.values.max()],
    'PC1 var%': [
        pca_rna.explained_variance_ratio_[0]*100,
        pca_dnam.explained_variance_ratio_[0]*100,
        pca_cnv.explained_variance_ratio_[0]*100
    ]
}).set_index('Modality').round(3)

summary